# Robot Görüş Modeli - Master Veri Seti Eğitimi
Bu notebook, birleştirilmiş **Master Veri Seti** kullanılarak eğitilecek model için hazırlanmıştır.

## Neler Otomatikleşti?
- **Yol Düzeltme:** Veri seti klasör yapısındaki kaymalar artık otomatik düzeltiliyor.
- **Drive Kaydı:** Eğitim sonuçları otomatik olarak Drive'a yedekleniyor.

## Adım Adım Yapmanız Gerekenler:
1. **Runtime -> Change runtime type** kısmından **T4 GPU** seçin.
2. Aşağıdaki hücreleri sırasıyla çalıştırın.

In [ ]:
# 1. GPU Kontrolü
!nvidia-smi

In [ ]:
# 2. Google Drive Bağlantısı
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Ultralytics Kurulumu
!pip install ultralytics

In [ ]:
# 4. Veri Setini (Master) Açma
!unzip -q /content/drive/MyDrive/master_dataset.zip -d /content/
print("Master veri seti açıldı!")

In [ ]:
# 5. YAML Dosyasını Hazırlama ve Yolu Düzeltme
import yaml
import os

# Drive'daki dosyayı kopyala
!cp /content/drive/MyDrive/master_data.yaml /content/master_data.yaml

# Klasör yapısındaki hatayı otomatik düzelt (path: /content yapıyoruz)
with open('/content/master_data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content'

with open('/content/master_data.yaml', 'w') as f:
    yaml.dump(data, f)

print("YAML dosyası kopyalandı ve klasör yolu başarıyla düzeltildi.")

In [ ]:
# 6. Eğitimi Başlat
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/master_data.yaml',
    epochs=50,
    imgsz=640,
    device=0,
    batch=16,
    project='/content/runs',
    name='robot_master_model'
)

In [ ]:
# 7. Sonuçları Drive'a Kaydet
import shutil
import os

source_dir = '/content/runs'
dest_dir = '/content/drive/MyDrive/robot_master_results'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
    print(f"Sonuçlar Drive'a kopyalandı: {dest_dir}")
else:
    print("Hata: Sonuçlar bulunamadı.")